# Credit Application Data Visualisation with Seaborn

Four visualisation tasks on a real credit card application dataset — demonstrating core Seaborn plot types and when to use each.

| Plot | Type | Purpose |
|---|---|---|
| Age vs employment length | Scatterplot | Bivariate relationship, filtered subset |
| Age distribution | Histplot + KDE | Univariate distribution shape |
| Income by family status | Catplot (boxplot) | Categorical comparison, ordered |
| Feature correlations | Heatmap | Full correlation matrix |

**Dataset:** [Credit card application records — Kaggle](https://www.kaggle.com/datasets/rikdifos/credit-card-approval-prediction)  
Download `application_record.csv` and place it in `data/` to run this notebook.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')

In [ ]:
df = pd.read_csv('data/application_record.csv')

print(f"Shape: {df.shape}")
df.head(3)

In [ ]:
df.info()
print()
df.describe()

---
## 1. Scatterplot — Age vs Employment Length

`DAYS_BIRTH` and `DAYS_EMPLOYED` are stored as negative integers (days before application date). Converting to positive values makes the axes intuitive.

**Filter:** Only include rows where `DAYS_EMPLOYED < 0` — positive values in this column are a data encoding for pensioners/unemployed and should be excluded from employment length analysis.

In [ ]:
# Filter to employed applicants only (negative DAYS_EMPLOYED = actually employed)
scatter_df = df[df['DAYS_EMPLOYED'] < 0].copy()
scatter_df['DAYS_EMPLOYED_POS'] = -scatter_df['DAYS_EMPLOYED'] / 365  # convert to years
scatter_df['DAYS_BIRTH_POS']    = -scatter_df['DAYS_BIRTH']    / 365

print(f"Filtered to {len(scatter_df):,} employed applicants ({len(scatter_df)/len(df):.1%} of total)")

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=scatter_df,
    x='DAYS_BIRTH_POS',
    y='DAYS_EMPLOYED_POS',
    hue='CODE_GENDER',
    alpha=0.4,
    s=15
)
plt.title('Age vs Years Employed (employed applicants only)')
plt.xlabel('Age (years)')
plt.ylabel('Years employed')
plt.tight_layout()
plt.show()

print("""
Observation: Employment length increases with age as expected.
The dense cluster at 0–5 years employed spans all ages — likely
job changers. There's a hard ceiling around 40–45 years employed
corresponding to career-long employees.
""")

---
## 2. Distribution Plot — Applicant Age

`sns.histplot` with `kde=True` overlays a kernel density estimate on the histogram — useful for seeing both the raw counts and the smoothed distribution shape simultaneously.

In [ ]:
df['AGE_YEARS'] = (-df['DAYS_BIRTH']) / 365

plt.figure(figsize=(10, 6))
sns.histplot(
    data=df,
    x='AGE_YEARS',
    kde=True,
    bins=40,
    color='steelblue'
)
plt.title('Age Distribution of Credit Applicants')
plt.xlabel('Age (years)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

print(f"Age range: {df['AGE_YEARS'].min():.0f} – {df['AGE_YEARS'].max():.0f} years")
print(f"Median age: {df['AGE_YEARS'].median():.1f} years")
print("""
The distribution is roughly uniform from 25–55 with a gradual decline
toward older applicants. No strong skew — most applicants are working-age adults.
""")

---
## 3. Categorical Plot — Income by Family Status

Filtering to the bottom 50% of incomes (below median) to focus on where family status differences are most visible. Groups are ordered by median income for interpretability.

In [ ]:
income_threshold = df['AMT_INCOME_TOTAL'].quantile(0.5)
bottom_half_df   = df[df['AMT_INCOME_TOTAL'] <= income_threshold].copy()

# Order categories by median income (ascending)
order_by_income = (
    bottom_half_df
    .groupby('NAME_FAMILY_STATUS')['AMT_INCOME_TOTAL']
    .median()
    .sort_values()
    .index.tolist()
)

g = sns.catplot(
    data=bottom_half_df,
    x='NAME_FAMILY_STATUS',
    y='AMT_INCOME_TOTAL',
    order=order_by_income,
    kind='box',
    height=5,
    aspect=2
)
g.set_axis_labels('Family Status', 'Annual Income')
g.fig.suptitle('Income Distribution by Family Status (bottom 50% of earners)', y=1.02)
plt.tight_layout()
plt.show()

print("""
Ordering by median reveals a clear income gradient across family statuses.
Civil marriage and single applicants show wider IQRs — more income variability
within those groups compared to married applicants.
""")

---
## 4. Correlation Heatmap

A heatmap of the Pearson correlation matrix for all numeric features. `annot=True` shows exact values in each cell. `FLAG_MOBIL` is dropped first — it's nearly constant (almost everyone has a mobile phone) so its correlations are meaningless.

In [ ]:
heatmap_df  = df.drop(columns=['FLAG_MOBIL'])
numeric_df  = heatmap_df.select_dtypes(include=['number'])
corr_matrix = numeric_df.corr()

plt.figure(figsize=(14, 10))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    linewidths=0.5
)
plt.title('Feature Correlation Matrix (numeric columns)', fontsize=14)
plt.tight_layout()
plt.show()

# Identify strongest correlations (excluding self-correlation)
corr_pairs = (
    corr_matrix
    .unstack()
    .drop_duplicates()
    .sort_values(key=abs, ascending=False)
)
# Remove self-correlations
corr_pairs = corr_pairs[corr_pairs != 1.0]
print("Strongest correlations (absolute value):")
print(corr_pairs.head(6).round(3))